# 08. 演習解答(付録)

本書(ODE Book)の各章末 **Exercises** の解答集です。考え方(数式・理由)を述べ、
可能なものは `ode_book` の関数で数値的に検証します。**まず自力で解いてから**読むことを勧めます。

| 対象 | 章 |
|---|---|
| 00 | 微分積分の基礎(両書共通) |
| 02–07 | ODE 各章 |

## 00. 微分積分の基礎

In [1]:
from ode_book import calculus
import sympy as sp

**00-1 割線→接線**: $f(x)=x^2$ の差の商は $\dfrac{(2+h)^2-2^2}{h}=4+h \to 4$。$h\to0$ で接線の傾き $f'(2)=4$。

In [2]:
for h in (1.0, 0.5, 0.1, 0.01):
    print(f"h={h:5.2f}  secant slope = {calculus.secant_slope(lambda x: x**2, 2.0, h):.4f}  (= 4 + h)")

h= 1.00  secant slope = 5.0000  (= 4 + h)
h= 0.50  secant slope = 4.5000  (= 4 + h)
h= 0.10  secant slope = 4.1000  (= 4 + h)
h= 0.01  secant slope = 4.0100  (= 4 + h)


**00-2 Riemann 和の規則**: midpoint は誤差 $O(\Delta x^2)$、left/right は $O(\Delta x)$。よって midpoint が最速で $1/3$ に収束。

In [3]:
f = lambda x: x**2
for n in (10, 100):
    row = {r: calculus.riemann_sum(f, 0, 1, n, r) for r in ("left", "mid", "right")}
    print(f"n={n:3d}  left={row['left']:.5f}  mid={row['mid']:.5f}  right={row['right']:.5f}  (exact 1/3={1/3:.5f})")

n= 10  left=0.28500  mid=0.33250  right=0.38500  (exact 1/3=0.33333)
n=100  left=0.32835  mid=0.33333  right=0.33835  (exact 1/3=0.33333)


**00-3 FTC**: $\int_0^x e^{-t}\,dt = 1-e^{-x}$。累積積分が解析解に一致する。

In [4]:
import numpy as np
xs = np.linspace(0, 3, 400)
F = calculus.cumulative_integral(lambda t: np.exp(-t), xs)
print("max |F - (1 - e^-x)| =", float(np.max(np.abs(F - (1 - np.exp(-xs))))))

max |F - (1 - e^-x)| = 4.4764733087010455e-06


**00-4 勾配**: $\nabla(\sin x\cos y)=(\cos x\cos y,\,-\sin x\sin y)$。$(\pi/4,\pi/4)$ では $(\tfrac12,-\tfrac12)$。

In [5]:
import numpy as np
f = lambda p: np.sin(p[0]) * np.cos(p[1])
g = calculus.gradient(f, [np.pi / 4, np.pi / 4])
print("numerical grad =", g, " | analytic = (0.5, -0.5)")

numerical grad = [ 0.5 -0.5]  | analytic = (0.5, -0.5)


**00-5 Taylor**: $\ln(1+x)=x-\tfrac{x^2}{2}+\tfrac{x^3}{3}-\tfrac{x^4}{4}+\tfrac{x^5}{5}-\cdots$。$x=0.5$ で 5 次近似の誤差を評価。

In [6]:
import numpy as np
x = sp.symbols("x")
poly = calculus.taylor_series(sp.log(1 + x), x, 0, 5)
print("Taylor(ln(1+x), order 5):", poly)
approx = calculus.taylor_approx(sp.log(1 + x), x, 0, 5)
print("error at x=0.5:", abs(float(approx(0.5)) - np.log(1.5)))

Taylor(ln(1+x), order 5): x**5/5 - x**4/4 + x**3/3 - x**2/2 + x
error at x=0.5: 0.001826558558502278


In [7]:
# Shared setup. Make the book package importable whether or not it is pip-installed,
# then fix the random seed and tidy NumPy printing.
import sys
from pathlib import Path

try:
    import ode_book  # noqa: F401
except ModuleNotFoundError:
    for _base in (Path.cwd(), *Path.cwd().parents):
        if (_base / "src" / "ode_book").is_dir():
            sys.path.insert(0, str(_base / "src"))
            break

import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
np.set_printoptions(precision=4, suppress=True)

In [8]:
from ode_book import systems, solvers
import numpy as np

## 02. 一階 ODE

**02-1**: $dy/dt=-ky$ は変数分離で $y=y_0e^{-kt}$。半減期は $y(t_{1/2})=y_0/2 \Rightarrow e^{-kt_{1/2}}=1/2 \Rightarrow t_{1/2}=\ln 2/k$、
$y_0$ に依らない。

In [9]:
k = 0.7
t_half = np.log(2) / k
for y0 in (1.0, 5.0, 100.0):
    print(f"y0={y0:6.1f}  y(t_half)/y0 = {np.exp(-k * t_half):.4f}  (= 1/2 for any y0)")
print("half-life =", t_half, "(independent of y0)")

y0=   1.0  y(t_half)/y0 = 0.5000  (= 1/2 for any y0)
y0=   5.0  y(t_half)/y0 = 0.5000  (= 1/2 for any y0)
y0= 100.0  y(t_half)/y0 = 0.5000  (= 1/2 for any y0)
half-life = 0.9902102579427791 (independent of y0)


**02-2**: $y'+2y=e^{-t}$。積分因子 $\mu=e^{2t}$ で $(e^{2t}y)'=e^{t}$、よって $y=e^{-t}+Ce^{-2t}$。

In [10]:
import sympy as sp
t = sp.symbols("t")
y = sp.Function("y")
print(sp.dsolve(sp.Eq(y(t).diff(t) + 2 * y(t), sp.exp(-t)), y(t)))

Eq(y(t), (C1*exp(-t) + 1)*exp(-t))


**02-3**: ロジスティックで $y_0>K$ から始めると $dy/dt=ry(1-y/K)<0$ なので **上から単調減少** して $K$ に収束。

In [11]:
f = systems.logistic(r=0.9, K=1.0)
t = np.linspace(0, 12, 300)
Y = solvers.rk4(f, [1.7], t)[:, 0]
print("monotone decreasing:", bool(np.all(np.diff(Y) < 0)), "| final ->", round(float(Y[-1]), 4), "(K=1)")

monotone decreasing: True | final -> 1.0 (K=1)


**02-4**: $dy/dt=\sqrt{|y|},\,y(0)=0$ は Lipschitz でない。$y\equiv 0$ と $y=(t/2)^2$ がともに解
($\dot y=t/2=\sqrt{(t/2)^2}=\sqrt{y}$)。一意性が破れる。

In [12]:
t = np.linspace(0, 4, 200)
y_a = np.zeros_like(t)
y_b = (t / 2) ** 2
# both satisfy dy/dt = sqrt(|y|)
res_b = np.max(np.abs(np.gradient(y_b, t)[1:-1] - np.sqrt(np.abs(y_b))[1:-1]))
print("y=0 is a solution; y=(t/2)^2 residual ~", round(float(res_b), 4), "-> two solutions from y(0)=0")

y=0 is a solution; y=(t/2)^2 residual ~ 0.0 -> two solutions from y(0)=0


## 03. 線形 ODE と連立系

**03-1**: $\ddot x+5\dot x+6x=0 \Rightarrow \begin{pmatrix}0&1\\-6&-5\end{pmatrix}$。固有値 $-2,-3$(実・相異・負)→ 振動しない **過減衰**。

In [13]:
A = np.array([[0.0, 1.0], [-6.0, -5.0]])
print("eigenvalues:", np.linalg.eigvals(A), "->", systems.classify_fixed_point(A), "(overdamped: real, distinct, negative)")

eigenvalues: [-2. -3.] -> stable node (overdamped: real, distinct, negative)


**03-2**: 無減衰調和振動子のエネルギー $E=\tfrac12(v^2+\omega^2x^2)$。Euler 法は増幅率 $>1$ で **エネルギーが増大**(RK4 はほぼ保存)。

In [14]:
f = systems.harmonic_oscillator(omega=1.0, gamma=0.0)
t = np.linspace(0, 20, 400)
for name, Y in [("Euler", solvers.euler(f, [1.0, 0.0], t)), ("RK4", solvers.rk4(f, [1.0, 0.0], t))]:
    E = 0.5 * (Y[:, 1] ** 2 + Y[:, 0] ** 2)
    print(f"{name:5s}: energy E(0)={E[0]:.3f} -> E(T)={E[-1]:.3f}  (drift {E[-1] - E[0]:+.3f})")

Euler: energy E(0)=0.500 -> E(T)=1.361  (drift +0.861)
RK4  : energy E(0)=0.500 -> E(T)=0.500  (drift -0.000)


**03-3**: $A(\Omega)=F_0/\sqrt{(\omega^2-\Omega^2)^2+(2\gamma\Omega)^2}$。分母 $g(u)=(\omega^2-u)^2+4\gamma^2u$($u=\Omega^2$)を最小化:
$g'(u)=-2(\omega^2-u)+4\gamma^2=0 \Rightarrow u=\omega^2-2\gamma^2$。よって $\Omega^2=\omega^2-2\gamma^2$。

In [15]:
w, g = 1.0, 0.2
Om = np.linspace(0.2, 1.5, 4000)
A = 1.0 / np.sqrt((w**2 - Om**2) ** 2 + (2 * g * Om) ** 2)
print("numerical peak Omega^2 =", round(float(Om[np.argmax(A)] ** 2), 4),
      " | formula w^2-2gamma^2 =", round(w**2 - 2 * g**2, 4))

numerical peak Omega^2 = 0.9198  | formula w^2-2gamma^2 = 0.92


## 04. 相図と安定性

**04-1**: $A=\begin{pmatrix}2&1\\1&2\end{pmatrix}$。固有値 $\lambda=2\pm1=\{3,1\}$(実・正)→ **unstable node**。

In [16]:
A = np.array([[2.0, 1.0], [1.0, 2.0]])
print("eigenvalues:", np.linalg.eigvals(A), "->", systems.classify_fixed_point(A))

eigenvalues: [3. 1.] -> unstable node


**04-2**: 減衰振動子 $\begin{pmatrix}0&1\\-\omega^2&-2\gamma\end{pmatrix}$。$0<\gamma<\omega$ で固有値は実部 $-\gamma$ の複素共役 → **stable spiral**。

In [17]:
w, g = 1.0, 0.3
J = np.array([[0.0, 1.0], [-(w**2), -2 * g]])
print("eigenvalues:", np.round(np.linalg.eigvals(J), 3), "->", systems.classify_fixed_point(J))

eigenvalues: [-0.3+0.954j -0.3-0.954j] -> stable spiral


**04-3**: 競合系の内部固定点 $(1,1)$ のヤコビ行列は $\begin{pmatrix}-1&-2\\-1&-1\end{pmatrix}$。$\det=-1<0$ → **saddle**。
鞍点は不安定なので 2 種は安定共存できない(**競争排除**)。

In [18]:
def fcomp(t, s):
    x, y = s
    return np.array([x * (3 - x - 2 * y), y * (2 - x - y)])

J = systems.jacobian(fcomp, [1.0, 1.0])
print("Jacobian at (1,1):\n", np.round(J, 3))
print("det =", round(float(np.linalg.det(J)), 3), "->", systems.classify_fixed_point(J), "(no stable coexistence)")

Jacobian at (1,1):
 [[-1. -2.]
 [-1. -1.]]
det = -1.0 -> saddle (no stable coexistence)


## 05. 非線形ダイナミクス

**05-1**: Lotka-Volterra の保存量 $V=\delta x-\gamma\ln x+\beta y-\alpha\ln y$ は軌道上で一定。

In [19]:
al, be, de, ga = 1.1, 0.4, 0.1, 0.4
f = systems.lotka_volterra(al, be, de, ga)
t = np.linspace(0, 24, 4000)
Y = solvers.rk4(f, [2.0, 1.0], t)
V = de * Y[:, 0] - ga * np.log(Y[:, 0]) + be * Y[:, 1] - al * np.log(Y[:, 1])
print("V range over the orbit:", round(float(V.max() - V.min()), 6), "(~0 => conserved)")

V range over the orbit: 0.0 (~0 => conserved)


**05-2**: SIR の $\dot I=\beta SI-\gamma I=I(\beta S-\gamma)$。ピーク($\dot I=0,\,I>0$)は $S=\gamma/\beta$。

In [20]:
beta, gamma = 0.5, 0.2
f = systems.sir(beta, gamma, 1.0)
t = np.linspace(0, 60, 4000)
Y = solvers.rk4(f, [0.99, 0.01, 0.0], t)
S_at_peak = Y[np.argmax(Y[:, 1]), 0]
print("S at infection peak =", round(float(S_at_peak), 4), " | gamma/beta =", gamma / beta)

S at infection peak = 0.4002  | gamma/beta = 0.4


**05-3**: $\dot y=ry-y^3=g(y)$、$g'(y)=r-3y^2$。$r>0$ で $g'(0)=r>0$ → $y=0$ 不安定;
$g'(\pm\sqrt r)=r-3r=-2r<0$ → $y=\pm\sqrt r$ 安定。

In [21]:
r = 0.5
gp = lambda y: r - 3 * y**2
print("g'(0) =", gp(0), "(>0 unstable)   g'(+/-sqrt r) =", round(gp(np.sqrt(r)), 3), "(<0 stable)")

g'(0) = 0.5 (>0 unstable)   g'(+/-sqrt r) = -1.0 (<0 stable)


**05-4**: Lorenz の初期差を $10^{-6}$ にすると、$10^{-8}$ より **早く** 分岐する(分離時刻 $\approx \frac1\lambda\ln(\text{閾値}/\text{初期差})$、初期差が大きいほど早い)。

In [22]:
f = systems.lorenz()
t = np.linspace(0, 40, 8000)
base = solvers.solve(f, [1.0, 1.0, 1.0], t, rtol=1e-9, atol=1e-9)
for gap in (1e-6, 1e-8):
    pert = solvers.solve(f, [1.0, 1.0, 1.0 + gap], t, rtol=1e-9, atol=1e-9)
    d = np.abs(base[:, 0] - pert[:, 0])
    t_sep = t[np.argmax(d > 1.0)]
    print(f"gap={gap:.0e}: trajectories separate (|dx|>1) at t ~ {t_sep:.1f}")

gap=1e-06: trajectories separate (|dx|>1) at t ~ 24.5
gap=1e-08: trajectories separate (|dx|>1) at t ~ 30.3


## 06. 数値解法

**06-1**: Heun の収束次数(log-log の傾き)を測ると $\approx 2$。

In [23]:
f = systems.exponential(-1.0)
dts, errs = [], []
for n in (16, 32, 64, 128, 256):
    t = np.linspace(0, 2, n + 1)
    dts.append(t[1] - t[0])
    errs.append(solvers.global_error(solvers.heun(f, [1.0], t)[:, 0], np.exp(-t)))
print("measured Heun order (slope) =", round(float(np.polyfit(np.log(dts), np.log(errs), 1)[0]), 3))

measured Heun order (slope) = 2.031


**06-2**: 陽的 Euler を $\dot y=\lambda y$ に使うと $y_{k+1}=(1+\lambda\Delta t)y_k$。安定は $|1+\lambda\Delta t|<1$。
$\lambda<0$(実)では $-1<1+\lambda\Delta t \Rightarrow \lambda\Delta t>-2 \Rightarrow \Delta t<2/|\lambda|$。

In [24]:
lam = -50.0
print("stability bound dt < 2/|lambda| =", 2 / abs(lam))
for dt in (0.039, 0.041):
    print(f"  amplification |1+lambda*dt| at dt={dt}: {abs(1 + lam * dt):.3f} ({'stable' if abs(1 + lam * dt) < 1 else 'UNSTABLE'})")

stability bound dt < 2/|lambda| = 0.04
  amplification |1+lambda*dt| at dt=0.039: 0.950 (stable)
  amplification |1+lambda*dt| at dt=0.041: 1.050 (UNSTABLE)


**06-3**: stiff 系では陽的 RK45 は安定性のため小刻みを強いられ `nfev` が爆発、陰的 Radau は大刻みで安定。

In [25]:
from scipy.integrate import solve_ivp
A = np.array([[-1.0, 0.0], [0.0, -1000.0]])
f = systems.linear_system(A)
nfev = {m: solve_ivp(f, (0, 5), [1.0, 1.0], method=m, rtol=1e-6, atol=1e-9).nfev for m in ("RK45", "Radau")}
print(nfev, "-> RK45/Radau nfev ratio =", round(nfev["RK45"] / nfev["Radau"], 1))

{'RK45': 10796, 'Radau': 1022} -> RK45/Radau nfev ratio = 10.6


## 07. 応用

**07-1**: 強制振動子の定常振幅は $\Omega\approx\omega$ でピーク(弱減衰なら $\Omega^2=\omega^2-2\gamma^2\approx\omega^2$)。

In [26]:
w, g, F0 = 1.0, 0.1, 0.5
Om = np.linspace(0.2, 2.0, 4000)
A = F0 / np.sqrt((w**2 - Om**2) ** 2 + (2 * g * Om) ** 2)
print("peak near Omega =", round(float(Om[np.argmax(A)]), 3), "(omega = 1)")

peak near Omega = 0.99 (omega = 1)


**07-2**: Vasicek の決定論部分 $\dot r=\kappa(\theta-r)$。$z=r-\theta$ とおくと $\dot z=-\kappa z \Rightarrow z=z_0e^{-\kappa t}$、
よって $r(t)=\theta+(r_0-\theta)e^{-\kappa t}$。$\kappa$ が大きいほど速く $\theta$ へ回帰。

In [27]:
theta, r0 = 0.03, 0.09
t = np.linspace(0, 10, 200)
for kappa in (0.3, 1.5):
    r = theta + (r0 - theta) * np.exp(-kappa * t)
    half = t[np.argmax(np.abs(r - theta) < 0.5 * abs(r0 - theta))]
    print(f"kappa={kappa}: half-way-to-theta time ~ {half:.2f}  (larger kappa -> faster)")

kappa=0.3: half-way-to-theta time ~ 2.31  (larger kappa -> faster)
kappa=1.5: half-way-to-theta time ~ 0.50  (larger kappa -> faster)


**07-3**: フィードバック $\dot x=(a-k)x$ は $a-k<0 \Leftrightarrow k>a$ で安定。$a=2$ なら **最小ゲイン $k=2$**(厳密には $k>2$)。

In [28]:
a = 2.0
print("stabilizing requires k > a =", a, "-> minimal gain k_min =", a)

stabilizing requires k > a = 2.0 -> minimal gain k_min = 2.0
